# Floating point numbers

**Lecture 4: Numbers and NumPy, notebook 1 of 2**

Computers store only finitely many decimal values, so calculated floats are usually approximations. The practical lesson is simple: compare computed floats with a tolerance rather than exact equality.

**Table of contents**<a id='toc0_'></a>    
- 1. [Floats are approximations](#toc1_)    
- 2. [Why `==` can surprise you](#toc2_)    
- 3. [Comparing floats correctly](#toc3_)    
- 4. [`inf` and `nan`](#toc4_)    
- 5. [Try it yourself](#toc5_)    
- 6. [Summary](#toc6_)    
- 7. [[Optional] If you are curious](#toc7_)    
  - 7.1. [How a float is stored](#toc7_1_)    
  - 7.2. [Underflow](#toc7_2_)    
  - 7.3. [Order of additions](#toc7_3_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## 1. <a id='toc1_'></a>[Floats are approximations](#toc0_)

Not all decimal numbers can be represented ***exactly*** on a computer. A *close* neighboring number is used instead:

In [1]:
x = 0.1
print(x)             # Python prints the short version
print(f'{x:.50f}')   # printing x with 50 decimals
x = 17.2
print(f'{x:.50f}')

0.1
0.10000000000000000555111512312578270211815834045410
17.19999999999999928945726423989981412887573242187500


<details>
<summary><strong>Why extra digits appear</strong></summary>

The computer stores numbers in base 2. Just as $1/3 = 0.333\ldots$ cannot be written with finitely many decimal digits, `0.1` and `17.2` cannot be written with finitely many binary digits. Python stores the nearest available number (about 16 significant digits). Ordinary printing hides the difference, while formatting with 50 decimals makes it visible.

</details>

The error sits around the 17th digit. For economic calculations it is irrelevant, **until you compare numbers**.

## 2. <a id='toc2_'></a>[Why `==` can surprise you](#toc0_)

Arithmetic on these approximate numbers inherits the small errors. The classic example:

In [2]:
print(0.1 + 0.2)        # not exactly 0.3
print(0.1 + 0.2 == 0.3) # False

0.30000000000000004
False


Small errors also accumulate in loops:

In [ ]:
d = 0.0
for i in range(10):
    d += 0.1
print(d)
print(d == 1.0)
print(d - 1.0)

0 0.0
1 0.1
2 0.2
3 0.30000000000000004
4 0.4
5 0.5
6 0.6
7 0.7
8 0.7999999999999999
9 0.8999999999999999
0.9999999999999999
False
-1.1102230246251565e-16


The difference is about $10^{-16}$: economically nothing, but enough to make `==` fail.

## 3. <a id='toc3_'></a>[Comparing floats correctly](#toc0_)

**The fix** is to compare with a *tolerance* instead of exact equality:

- `x == y` asks: are the two stored numbers **exactly identical**?
- `math.isclose(x, y)` asks: are they **close enough**?

In [4]:
import math

x = 0.1 + 0.2
y = 0.3
print(x == y)             # False
print(math.isclose(x, y)) # True: equal within a tolerance

False
True


<details>
<summary><strong>What `math.isclose` asks</strong></summary>

By default, `math.isclose(x, y)` uses a **relative** tolerance `rel_tol=1e-09`: `x` and `y` count as close if they differ by less than a billionth of the larger of the two. You can change it, e.g. `math.isclose(x, y, rel_tol=1e-6)`. See the [Python documentation](https://docs.python.org/3/library/math.html#math.isclose).

</details>

**One trap: comparing with zero.** Comparing with zero is special: a relative tolerance alone does not give us a useful fixed neighbourhood around zero. We therefore add an absolute tolerance `abs_tol`:

In [5]:
residual = 0.1 + 0.2 - 0.3
print(residual)
print(math.isclose(residual, 0))                 # False!
print(math.isclose(residual, 0, abs_tol=1e-12))  # True

5.551115123125783e-17
False
True


**Rule of thumb:**
- use `==` for integers and strings
- use `math.isclose(x, y)` for floats that come out of calculations
- when comparing with `0`, add `abs_tol`, e.g. `abs_tol=1e-12`

## 4. <a id='toc4_'></a>[`inf` and `nan`](#toc0_)

**Overflow**: a calculation can be mathematically correct and still fail numerically, because an *intermediate* value becomes too large to store:

In [6]:
x = 1.0
y = 2.7
for i in range(200):
    x *= (i+1)
    y *= (i+1)
print(y/x) # should be 2.7
print(x,y)

nan
inf inf


**What happened?** Both numbers are multiplied by the same factors $1, 2, \ldots, 200$, so mathematically

$$\frac{y}{x} = \frac{2.7 \cdot 1 \cdot 2 \cdots 200}{1 \cdot 1 \cdot 2 \cdots 200} = \frac{2.7 \cdot 200!}{200!} = 2.7.$$

- **On paper**, we cancel $200!$ first and get $2.7$.
- **The computer** builds both huge numbers first. But $200! \approx 7.9 \cdot 10^{374}$, far above the largest float ($\approx 1.8 \cdot 10^{308}$). So `x` and `y` both become `inf`: *too large to represent*.
- The last step is then `inf/inf`. This has no unique answer (it could come from $10^{400}/10^{400} = 1$ or from $2.7 \cdot 10^{400}/10^{400} = 2.7$), so Python returns `nan`: *not a number*.

**Lesson:** the formula is fine; the problem is the intermediate values.

**Note:** `inf` is infinite. `nan` is *not a number*.

`nan` is not equal to anything, **not even to itself**. Test for it with `math.isnan()`, never with `==`:

In [7]:
z = y/x
print(z == z)          # False!
print(math.isnan(z))   # True

False
True


If `inf` or `nan` shows up in your results, it almost always signals a problem worth investigating. You will meet `nan` again when we work with data, where it marks missing values.

## 5. <a id='toc5_'></a>[Try it yourself](#toc0_)

**Try it yourself, 5 minutes:** A household spends its income on three categories with budget shares

| rent | consumption | savings |
|---|---|---|
| 0.3 | 0.6 | 0.1 |

The shares should sum to one.

1. Compute `total`, the sum of the three shares, and check whether it equals `1.0` using `==`.
2. Check the same with `math.isclose()`.
3. Compute the residual `total - 1.0` and check whether it is zero with `math.isclose()`, first without and then with `abs_tol=1e-12`. Why do the two answers differ?

## 6. <a id='toc6_'></a>[Summary](#toc0_)

The take-aways are:

1. Decimal numbers are **approximate** on a computer. Tiny differences after calculations are normal.
2. **Never test calculated floats for exact equality** with `==`. Use `math.isclose` (a comparison with a tolerance). When comparing with `0`, set `abs_tol`.
3. `inf` and `nan` signal that something in a calculation went wrong. Test for `nan` with `math.isnan()`.

**In practice:** for everyday economics code you mainly need rule 2: reach for `math.isclose` whenever you compare computed numbers.

For further details see [here](https://docs.python.org/3/tutorial/floatingpoint.html).

**Videos:**

* [Why computers are bad at algebra - Infinite Series](https://www.youtube.com/watch?v=pQs_wx8eoQ8)
* [Floating point numbers - Computerphile](https://www.youtube.com/watch?v=PZRI1IfStY0)

## 7. <a id='toc7_'></a>[[Optional] If you are curious](#toc0_)

Not needed for the course. Read this if you want to know what happens under the hood.

### 7.1. <a id='toc7_1_'></a>[How a float is stored](#toc0_)

On a computer the real line is approximated with numbers on the form:

$$\text{number} = \text{sign} \times \text{significand} \times \text{base}^{\text{exponent}}$$

For a standard 64-bit number the `base` is 2 and the bits are allocated as:
* **sign**: 1 bit (positive or negative)
* **exponent**: 11 bits
* **significand**: 52 bits

52 binary digits correspond to roughly 15 to 17 significant decimal digits.

### 7.2. <a id='toc7_2_'></a>[Underflow](#toc0_)

Multiplying many small numbers can result in an exact zero:

In [8]:
x = 1e-60
y = 1
for _ in range(6):
    y *= x
    print(y)

1e-60
1e-120
1e-180
1e-240
9.999999999999999e-301
0.0


### 7.3. <a id='toc7_3_'></a>[Order of additions](#toc0_)

On paper, addition does not depend on the order. With floats it does, but not by much:

In [9]:
sum1 = 10001234.0 + 0.12012 + 0.12312 + 1e-5
sum2 = 1e-5 + 0.12312 + 0.12012 + 10001234.0
print(sum1-sum2)
print(math.isclose(sum1, sum2))

1.862645149230957e-09
True
